# Fast Whisper — GPU Inference Optimization
Run top to bottom in **Colab (GPU runtime)** or a **Lightning AI CUDA notebook**. The project compares target-only, existing cached fixed drafting, buffered fixed drafting and adaptive drafting. Start with a smoke check, then save the measured evaluation for résumé wording. Model loading is excluded from latency. English speech, ≤30 seconds, batch size one.


In [ ]:
import sys, subprocess, tempfile, os
from pathlib import Path
import torch
assert torch.cuda.is_available(), "Enable a CUDA GPU runtime before continuing."
print(torch.cuda.get_device_name(0), torch.__version__)
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB")
if "REPO_DIR" not in globals():
    REPO_DIR = Path(tempfile.mkdtemp(prefix="fast-whisper-")) / "repo"
    subprocess.run(["git", "clone", "--single-branch", "--branch", "main",
                    "https://github.com/AarushRathore1729/fast-whisper.git", str(REPO_DIR)], check=True)
print("Project:", REPO_DIR)


In [ ]:
# Preserve preinstalled CUDA PyTorch. Install the tested Transformers release.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                f"{REPO_DIR}[dev,benchmark,demo,api]", "transformers==4.57.1"], check=True)
# CLI tests and benchmarks run in fresh processes.
sys.path.insert(0, str(REPO_DIR / "src"))
def run(*args):
    subprocess.run([sys.executable, *args], cwd=REPO_DIR, check=True)
run("-m", "pytest")


### 1. Two-clip GPU check
Downloads Large-v3 and Tiny once. All measured paths must match the target greedy decoder exactly. A mismatch saves evidence and stops; do not claim equivalence or speedup from a failed run.


In [ ]:
run("-m", "specvoice.compare", "--audio", "samples/audio1.wav", "samples/audio2.wav",
    "--max-samples", "2", "--repetitions", "1", "--warmup-runs", "1",
    "--max-new-tokens", "128", "--output", "results/gpu_smoke.json")


### 2. Measured evaluation
Ten LibriSpeech test utterances, three repeats, randomized path order. CUDA synchronization surrounds each component measurement. The profiler runs separately. Increase `--max-new-tokens` to 256 if any result is truncated. Optional `--verify-generate` adds an untimed Hugging Face oracle check.


In [ ]:
run("-m", "specvoice.compare", "--max-samples", "10", "--repetitions", "3",
    "--warmup-runs", "1", "--max-new-tokens", "128", "--profile",
    "--no-cached-suppression", "--output", "results/comparison.json")


In [ ]:
import json
from IPython.display import display, Markdown
report = json.loads((REPO_DIR / "results/comparison.json").read_text())
assert report["status"] == "complete" and report["all_exact_token_matches"]
display(Markdown((REPO_DIR / "results/comparison.md").read_text()))
print(json.dumps(report["environment"], indent=2))
print("Truncated runs:", {m: s["truncated_runs"] for m, s in report["summary"].items()})
print("Keep comparison.json for the final résumé bullets.")


### 2b. Final validated cached-mask evaluation
The recorded 30-utterance T4 run achieved 1.60× request speedup and 2.00× decoder speedup, with the Hugging Face oracle and all 90 token-digest checks passing. Cached masks are now the default for buffered paths. This cell reproduces the wider evaluation; section 2 retains the explicit uncached first-ten ablation. Clips above 30 seconds are logged and skipped.


In [ ]:
run("-m", "specvoice.compare", "--modes", "baseline", "optimized",
    "--max-samples", "30", "--sample-strategy", "evenly-spaced",
    "--repetitions", "3", "--warmup-runs", "1", "--max-new-tokens", "256",
    "--cached-suppression", "--verify-generate", "--profile",
    "--output", "results/expanded.json")
run("-m", "specvoice.audit", "results/expanded.json", "--output", "results/expanded.audit.json")
display(Markdown((REPO_DIR / "results/expanded.md").read_text()))


### 3. Audio-upload demonstration (optional)
The benchmark above is the priority. Lightning AI can open forwarded port 7860. Set `SHARE=True` only when you want a public temporary Gradio link (commonly needed in Colab). Stop the demo before rerunning GPU benchmarks.


In [ ]:
from specvoice.asr.transcriber import SpeculativeWhisperTranscriber
from specvoice.demo import build_demo
# If old Transformers modules were already imported, restart the kernel before
# this optional cell; preceding CLI benchmarks already used fresh processes.
SHARE = False
engine = SpeculativeWhisperTranscriber.from_pretrained()
demo = build_demo(engine)
demo.queue().launch(server_name="0.0.0.0", server_port=7860, share=SHARE, prevent_thread_lock=True)


### 4. Download the evidence
In Colab, the ZIP download starts automatically. On Lightning AI, download the ZIP from the printed path. Send `comparison.json` back to finalize the two résumé bullets using actual GPU measurements.


In [ ]:
import zipfile
archive = REPO_DIR / "fast-whisper-results.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for p in (REPO_DIR / "results").glob("*"):
        if p.is_file() and not p.name.endswith(".tmp"):
            z.write(p, p.name)
print(archive)
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:
    pass
